In [6]:
import os

print("Current folder:", os.getcwd())
print("Files here:", os.listdir())
print("Parent folder:", os.listdir(".."))

Current folder: /Users/stacychan/Query Analysis/src
Files here: ['.DS_Store', '02_behavior_relationships.ipynb', 'Reconciled_Data.csv', '03_behavior_transitions.ipynb', '00_data_preprocessing.ipynb', '01_dataset_overview.ipynb', '04_user_behavior_profiles.ipynb']
Parent folder: ['.DS_Store', 'raw_data', 'processed_data', 'README.md', '.git', 'src']


In [7]:
import pandas as pd

df = pd.read_excel("../raw_data/Query_Analysis.xlsx")

df.to_csv("Reconciled_Data.csv", index=False)


In [8]:
import numpy as np

file_path = "Reconciled_Data.csv"

# save outputs into processed_data folder
output_dir = "../processed_data"
os.makedirs(output_dir, exist_ok=True)

output_full = os.path.join(output_dir, "Reconciled_Data_Full.csv")
output_final = os.path.join(output_dir, "Reconciled_Final.csv")

# read raw csv
raw = pd.read_csv(file_path, header=None)

# actual data starts after the first 2 rows
df = raw.iloc[2:].copy().reset_index(drop=True)

# assign columns manually based on the file structure
df.columns = [
    "participant_no",
    "week_no",
    "query",

    "info_type_sophie",
    "info_type_ari",
    "info_type_bowen",
    "info_type_lexie",
    "info_type_agreement",
    "info_type_reconciliation",

    "task_sophie",
    "task_rachel",
    "task_bowen",
    "task_lexie",
    "task_agreement",
    "task_reconciliation",

    "goal_sophie",
    "goal_ari",
    "goal_bowen",
    "goal_lexie",
    "goal_agreement",
    "goal_reconciliation",
]

# turn empty strings into NA
df = df.replace(r"^\s*$", pd.NA, regex=True)

# fill merged-like blanks in participant/week
df["participant_no"] = df["participant_no"].ffill()
df["week_no"] = df["week_no"].ffill()


def is_yes(x):
    return pd.notna(x) and str(x).strip().upper() == "Y"


def fill_within_group(series):
    return series.ffill().bfill()


# --------------------------------------------------
# LABEL NORMALIZATION
# --------------------------------------------------
def normalize_label(x):

    if pd.isna(x):
        return pd.NA

    val = str(x).strip().lower()
    val = " ".join(val.split())

    label_map = {

        # procedural variants
        "how-to": "how-to (procedural)",
        "procedural": "how-to (procedural)",
        "procedual": "how-to (procedural)",
        "proc": "how-to (procedural)",
        "how-to (procedural)": "how-to (procedural)",

        # psychological variants
        "psyc": "how-to (psychological)",
        "psychological": "how-to (psychological)",
        "how-to (psyc)": "how-to (psychological)",
        "how-to (psychological)": "how-to (psychological)",

        # ideas
        "ideas": "ideas/options",
        "ideas/options": "ideas/options",

        # evaluation typos
        "evaluate": "evaluation",
        "evalute": "evaluation",
        "evaluation": "evaluation",

        # outcome
        "outcome": "outcome expectancy",
        "outcome expectancy": "outcome expectancy",

        # barrier (leave k unchanged)
        "barrier": "barrier management",
        "barrier management": "barrier management",

         # task mappings you requested
        "action": "action",

        "decision": "decision making",
        "decision making": "decision making",

        "motiv": "motivational reasoning",
        "motivational": "motivational reasoning",
        "motivational reasoning": "motivational reasoning",

        "plan": "plan",
        "plna": "plan",

        # NA normalization
        "na": "na"
    }

    return label_map.get(val, val)


def first_mode(row_vals):

    vals = [normalize_label(v) for v in row_vals if pd.notna(v) and str(v).strip() != ""]
    vals = [v for v in vals if pd.notna(v)]

    if not vals:
        return pd.NA

    return pd.Series(vals).value_counts().index[0]


group_keys = ["participant_no", "week_no", "query"]

# --------------------------------------------------
# STANDARDIZE CODER LABELS FIRST
# --------------------------------------------------

label_cols = [
    "info_type_sophie", "info_type_ari", "info_type_bowen", "info_type_lexie", "info_type_reconciliation",
    "task_sophie", "task_rachel", "task_bowen", "task_lexie", "task_reconciliation",
    "goal_sophie", "goal_ari", "goal_bowen", "goal_lexie", "goal_reconciliation",
]

for col in label_cols:
    df[col] = df[col].apply(normalize_label)


# --------------------------------------------------
# 1) fill reconciliation within repeated query rows
# --------------------------------------------------

for col in [
    "info_type_reconciliation",
    "task_reconciliation",
    "goal_reconciliation",
]:
    df[col] = df.groupby(group_keys, dropna=False)[col].transform(fill_within_group)


# --------------------------------------------------
# 2) if agreed = N but reconciliation blank
#    use majority vote
# --------------------------------------------------

# info type fallback
mask = (~df["info_type_agreement"].apply(is_yes)) & (df["info_type_reconciliation"].isna())

df.loc[mask, "info_type_reconciliation"] = df.loc[
    mask,
    ["info_type_lexie", "info_type_bowen", "info_type_ari", "info_type_sophie"]
].apply(first_mode, axis=1)


# task fallback
mask = (~df["task_agreement"].apply(is_yes)) & (df["task_reconciliation"].isna())

df.loc[mask, "task_reconciliation"] = df.loc[
    mask,
    ["task_lexie", "task_bowen", "task_rachel", "task_sophie"]
].apply(first_mode, axis=1)


# goal fallback
mask = (~df["goal_agreement"].apply(is_yes)) & (df["goal_reconciliation"].isna())

df.loc[mask, "goal_reconciliation"] = df.loc[
    mask,
    ["goal_sophie", "goal_lexie", "goal_bowen", "goal_ari"]
].apply(first_mode, axis=1)


# --------------------------------------------------
# 3) apply final rule
# --------------------------------------------------

df["final_type"] = np.where(
    df["info_type_agreement"].apply(is_yes),
    df["info_type_lexie"],
    df["info_type_reconciliation"]
)

df["final_task"] = np.where(
    df["task_agreement"].apply(is_yes),
    df["task_lexie"],
    df["task_reconciliation"]
)

df["final_goal"] = np.where(
    df["goal_agreement"].apply(is_yes),
    df["goal_sophie"],
    df["goal_reconciliation"]
)


# --------------------------------------------------
# 4) final safety fill across repeated rows
# --------------------------------------------------

for col in ["final_type", "final_task", "final_goal"]:
    df[col] = df.groupby(group_keys, dropna=False)[col].transform(fill_within_group)


# --------------------------------------------------
# 5) final normalization pass
# --------------------------------------------------

for col in ["final_type", "final_task", "final_goal"]:
    df[col] = df[col].apply(normalize_label)


# --------------------------------------------------
# SAVE OUTPUTS
# --------------------------------------------------

df.to_csv(output_full, index=False)

result = df[
    ["participant_no", "week_no", "query", "final_type", "final_task", "final_goal"]
].copy()

result.to_csv(output_final, index=False)


print("\nMissing values:")
print(result.isna().sum())

print("\nUnique final_type labels:")
print(sorted(result["final_type"].dropna().unique()))


Missing values:
participant_no    0
week_no           0
query             0
final_type        0
final_task        0
final_goal        0
dtype: int64

Unique final_type labels:
['comparative', 'experiential', 'factual', 'how-to (procedural)', 'how-to (psychological)', 'ideas/options', 'mechanism', 'na', 'normative', 'outcome expectancy', 'utility']
